# Volunteer Funnel

This notebook tracks session reach across key volunteer pages and the two main conversion endpoints:
- Digital Support signup clicks to `events.teams.microsoft.com`
- Phone Crisis Support form submissions on the final Drupal webform page

The notebook is session-based for funnel interpretation, with event counts also included for the two conversion endpoints.

### Quick start
1. Run the **Setup (run once)** cell.
2. Review the page URLs and endpoint parameters in **Parameters**.
3. Run the query cell.
4. Review the branch funnels and the daily conversion trends.

**Metrics:** page sessions, conversion sessions, conversion events  
**Data source:** GA4 BigQuery export (`events_*`)


In [ ]:
#@title Setup (run once)
import sys
import os

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q 'plotly>=6.1.1' 'kaleido>=1.2.0'
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import (
    build_date_params,
    default_query_window,
    dry_run_bytes,
    get_client,
    load_sql_template,
    run_query,
)

lifeline_theme.inject_fonts()

client = get_client()


In [ ]:
#@title Parameters
DAYS_BACK = 92 #@param {type:"integer"}

VOLUNTEER_PAGE = "https://www.lifeline.org.au/get-involved/volunteer" #@param {type:"string"}
CRISIS_SUPPORTER_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter" #@param {type:"string"}
PHONE_SUPPORT_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter/phone-crisis-support" #@param {type:"string"}
DIGITAL_SUPPORT_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter/digital-support" #@param {type:"string"}
PHONE_FORM_PAGE = "https://www.lifeline.org.au/get-involved/volunteer/volunteer-as-a-crisis-supporter/phone-crisis-support/phone-crisis-support" #@param {type:"string"}

DIGITAL_OUTBOUND_DOMAIN = "events.teams.microsoft.com" #@param {type:"string"}
PHONE_FORM_ID = "webform-submission-volunteer-form-paragraph-3216-add-form" #@param {type:"string"}

window = default_query_window(days_back=DAYS_BACK)
analysis_start = window.start_date
analysis_end = window.end_date
analysis_window_label = f"{analysis_start:%Y-%m-%d} to {analysis_end:%Y-%m-%d}"

print(f"Project: {config.PROJECT_ID}")
print(f"GA4 dataset: {config.GA4_DATASET}")
print(f"Window: {analysis_start} to {analysis_end}")
print(f"Phone form ID: {PHONE_FORM_ID}")


In [ ]:
query = load_sql_template(
    "sql/notebooks/ga4_volunteer_funnel_daily.sql",
    project_id=config.PROJECT_ID,
    ga4_dataset=config.GA4_DATASET,
)

params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("volunteer_page", "STRING", VOLUNTEER_PAGE),
    bigquery.ScalarQueryParameter("crisis_supporter_page", "STRING", CRISIS_SUPPORTER_PAGE),
    bigquery.ScalarQueryParameter("phone_support_page", "STRING", PHONE_SUPPORT_PAGE),
    bigquery.ScalarQueryParameter("digital_support_page", "STRING", DIGITAL_SUPPORT_PAGE),
    bigquery.ScalarQueryParameter("phone_form_page", "STRING", PHONE_FORM_PAGE),
    bigquery.ScalarQueryParameter("digital_outbound_domain", "STRING", DIGITAL_OUTBOUND_DOMAIN),
    bigquery.ScalarQueryParameter("phone_form_id", "STRING", PHONE_FORM_ID),
]

estimated_bytes = dry_run_bytes(client, query, params=params)
print(f"Estimated bytes scanned: {estimated_bytes:,}")

df = run_query(client, query, params=params)
nonzero_df = df[df.drop(columns=["report_date"]).sum(axis=1) > 0].copy()

print(f"Rows returned: {len(df):,}")
if nonzero_df.empty:
    print("No non-zero rows in the selected window.")
    df.head()
else:
    first_nonzero_date = nonzero_df["report_date"].min()
    print(f"First non-zero date in window: {first_nonzero_date}")
    nonzero_df.head()


In [ ]:
totals = df.drop(columns=["report_date"]).sum()

digital_signup_rate = totals["digital_signup_sessions"] / totals["digital_support_page_sessions"] if totals["digital_support_page_sessions"] else 0
phone_submit_rate = totals["phone_form_submit_sessions"] / totals["phone_form_page_sessions"] if totals["phone_form_page_sessions"] else 0
overall_conversion_rate = totals["any_conversion_sessions"] / totals["crisis_supporter_page_sessions"] if totals["crisis_supporter_page_sessions"] else 0

summary = pd.DataFrame(
    [
        ["Volunteer page sessions", totals["volunteer_page_sessions"], "count"],
        ["Crisis supporter page sessions", totals["crisis_supporter_page_sessions"], "count"],
        ["Digital support page sessions", totals["digital_support_page_sessions"], "count"],
        ["Digital Teams signup sessions", totals["digital_signup_sessions"], "count"],
        ["Digital signup rate from digital page", digital_signup_rate, "rate"],
        ["Phone support page sessions", totals["phone_support_page_sessions"], "count"],
        ["Phone form page sessions", totals["phone_form_page_sessions"], "count"],
        ["Phone form submit sessions", totals["phone_form_submit_sessions"], "count"],
        ["Phone submit rate from form page", phone_submit_rate, "rate"],
        ["Sessions with any conversion", totals["any_conversion_sessions"], "count"],
        ["Overall conversion rate from crisis supporter page", overall_conversion_rate, "rate"],
        ["Digital Teams click events", totals["digital_signup_events"], "count"],
        ["Phone form submit events", totals["phone_form_submit_events"], "count"],
    ],
    columns=["metric", "value", "value_type"],
)

summary["formatted_value"] = summary.apply(
    lambda row: f"{row['value']:.1%}" if row["value_type"] == "rate" else f"{int(row['value']):,}",
    axis=1,
)
summary[["metric", "formatted_value"]]


In [ ]:
digital_reach = pd.DataFrame(
    {
        "stage": [
            "Volunteer page",
            "Crisis supporter page",
            "Digital support page",
            "Teams signup sessions",
        ],
        "sessions": [
            int(totals["volunteer_page_sessions"]),
            int(totals["crisis_supporter_page_sessions"]),
            int(totals["digital_support_page_sessions"]),
            int(totals["digital_signup_sessions"]),
        ],
    }
)

digital_reach["share_of_crisis"] = digital_reach["sessions"] / totals["crisis_supporter_page_sessions"] if totals["crisis_supporter_page_sessions"] else 0

digital_reach["label"] = digital_reach.apply(
    lambda row: f"{row['sessions']:,} sessions | {row['share_of_crisis']:.1%} of crisis-page sessions",
    axis=1,
)

phone_reach = pd.DataFrame(
    {
        "stage": [
            "Volunteer page",
            "Crisis supporter page",
            "Phone support page",
            "Phone form page",
            "Phone form submit sessions",
        ],
        "sessions": [
            int(totals["volunteer_page_sessions"]),
            int(totals["crisis_supporter_page_sessions"]),
            int(totals["phone_support_page_sessions"]),
            int(totals["phone_form_page_sessions"]),
            int(totals["phone_form_submit_sessions"]),
        ],
    }
)

phone_reach["share_of_crisis"] = phone_reach["sessions"] / totals["crisis_supporter_page_sessions"] if totals["crisis_supporter_page_sessions"] else 0
phone_reach["label"] = phone_reach.apply(
    lambda row: f"{row['sessions']:,} sessions | {row['share_of_crisis']:.1%} of crisis-page sessions",
    axis=1,
)


def make_stage_reach_chart(frame, title, color):
    fig = px.bar(
        frame,
        x="sessions",
        y="stage",
        orientation="h",
        text="label",
        template="lifeline",
        title=title,
    )
    fig.update_traces(marker_color=color, textposition="outside", cliponaxis=False)
    fig.update_layout(
        height=480 if len(frame) <= 4 else 540,
        margin=dict(l=220, r=140, t=90, b=60),
        xaxis_title="Sessions",
        yaxis_title="",
    )
    fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(frame["stage"].tolist())))
    lifeline_theme.add_lifeline_logo(fig)
    return fig

make_stage_reach_chart(
    digital_reach,
    f"Volunteer Stage Reach: Digital Branch ({analysis_window_label})",
    lifeline_theme.SUPPORT_BLUE,
).show()

make_stage_reach_chart(
    phone_reach,
    f"Volunteer Stage Reach: Phone Branch ({analysis_window_label})",
    lifeline_theme.OPTIMISM_ORANGE,
).show()



In [ ]:
conversion_trend = df.melt(
    id_vars="report_date",
    value_vars=["digital_signup_sessions", "phone_form_submit_sessions", "any_conversion_sessions"],
    var_name="metric",
    value_name="sessions",
)

conversion_trend["metric"] = conversion_trend["metric"].map(
    {
        "digital_signup_sessions": "Digital Teams signup sessions",
        "phone_form_submit_sessions": "Phone form submit sessions",
        "any_conversion_sessions": "Any conversion sessions",
    }
)

fig = px.line(
    conversion_trend,
    x="report_date",
    y="sessions",
    color="metric",
    template="lifeline",
    title=f"Volunteer Conversion Sessions Over Time ({analysis_window_label})",
    labels={"report_date": "Date", "sessions": "Sessions", "metric": "Metric"},
)
fig.update_layout(hovermode="x unified")
lifeline_theme.add_lifeline_logo(fig)
fig.show()


## Interpretation notes

- `Digital Teams signup sessions` are sessions with a `click` event from the Digital Support page to `events.teams.microsoft.com`.
- `Phone form submit sessions` are sessions with a `form_submit` event on the final phone registration page for the Drupal form ID `webform-submission-volunteer-form-paragraph-3216-add-form`.
- The broad custom event `crisis_volunteer_leads` is intentionally not used as the phone conversion endpoint here, because it appears on multiple volunteer pages and is not specific enough for submit completion.
- This is a reach-and-conversion funnel, not a strictly nested one. Users can land directly on deeper pages, so some deeper-stage session counts can exceed the top-level volunteer page.
